# Phase 2 — Colab Trainer (FT-Transformer + Deep SVDD on CIC-IDS2017)

Run this notebook on a **Google Colab T4 GPU runtime**.

Defaults target **[antedotee/dl-aml-cybersec-project](https://github.com/antedotee/dl-aml-cybersec-project)**. Use **Runtime → Run all** after authorizing Drive when prompted.

Pipeline:
1. Clone / sync this repo from GitHub (always matches `main`).
2. Install requirements.
3. Mount Google Drive (checkpoints survive disconnects).
4. Download CIC-IDS2017 (cached to Drive after the first run).
5. Train: MFM pretrain → Deep SVDD finetune.
6. Persist `ft_svdd_final.pt` + scaler + `history.json` to Drive.

Optional: edit **§0** only if you fork the repo, need a quick smoke test, or resume after an interruption.

## 0. Configure

In [ ]:
# Default repo (change only if you use a fork)
REPO_URL = 'https://github.com/antedotee/dl-aml-cybersec-project.git'
REPO_BRANCH = 'main'
DRIVE_SUBDIR = 'cps_ad_phase2'   # under /content/drive/MyDrive

HELD_OUT_FAMILY = 'Heartbleed'   # attack family held out for zero-day eval

# Colab-safe defaults (reduce RAM pressure).
LOW_RAM_MODE = True
EPOCHS_MFM = 120 if LOW_RAM_MODE else 200
EPOCHS_SVDD = 60 if LOW_RAM_MODE else 100
BATCH_SIZE = 512 if LOW_RAM_MODE else 1024
MAX_TRAIN_ROWS = 300000 if LOW_RAM_MODE else 0  # 0 = all benign train rows

# Optional: restrict loaded CIC day files for RAM-constrained sessions.
# Leave empty to use all available day files.
CIC_FILES = [
    'Monday-WorkingHours.pcap_ISCX.csv',
    'Tuesday-WorkingHours.pcap_ISCX.csv',
    'Wednesday-workingHours.pcap_ISCX.csv',
    'Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv',
] if LOW_RAM_MODE else []

# Resume after Colab disconnect (leave None for a fresh run)
RESUME_CKPT = None               # e.g. Path('/content/drive/MyDrive/cps_ad_phase2/checkpoints/ft_svdd_mfm_last.pt')
SKIP_MFM = False                 # True only with RESUME_CKPT pointing at a finished MFM checkpoint

## 1. Detect environment + clone repo

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print('IN_COLAB =', IN_COLAB)

REPO_LOCAL = Path('/content/dl-aml-cybersec-project')

if IN_COLAB:
    if REPO_LOCAL.exists():
        subprocess.run(['git', '-C', str(REPO_LOCAL), 'fetch', 'origin', REPO_BRANCH], check=True)
        subprocess.run(['git', '-C', str(REPO_LOCAL), 'reset', '--hard', f'origin/{REPO_BRANCH}'], check=True)
    else:
        subprocess.check_call(
            ['git', 'clone', '--depth', '1', '-b', REPO_BRANCH, REPO_URL, str(REPO_LOCAL)]
        )
    os.chdir(REPO_LOCAL)
    print('cwd =', Path.cwd())


In [ ]:
if IN_COLAB:
    # Install project deps first.
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'],
        cwd=str(Path.cwd()),
    )

    # Repair NumPy in Colab: uninstall stale builds, then reinstall a pinned wheel.
    # This avoids mixed/partial installs that can trigger import sanity failures.
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'numpy'], check=False)
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'numpy'], check=False)
    subprocess.check_call(
        [
            sys.executable,
            '-m',
            'pip',
            'install',
            '-q',
            '--no-cache-dir',
            '--upgrade',
            '--force-reinstall',
            'numpy==1.26.4',
        ],
        cwd=str(Path.cwd()),
    )

    # Restart exactly once per VM so the kernel loads the fresh NumPy binary cleanly.
    import os
    import importlib
    restart_marker = Path.cwd() / '.phase2_numpy_restart_done'
    if not restart_marker.exists():
        restart_marker.write_text('done\n')
        print('NumPy repaired. Restarting runtime once to load clean binaries...')
        os.kill(os.getpid(), 9)

    np = importlib.import_module('numpy')
    print('numpy =', np.__version__, 'from', np.__file__)


## 2. Mount Drive + decide checkpoint dir

In [ ]:
from pathlib import Path
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive') / DRIVE_SUBDIR
else:
    DRIVE_ROOT = Path.cwd() / 'models' / 'phase2_local'
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DATA_DIR = DRIVE_ROOT / 'cic_ids'
OUT_DIR = DRIVE_ROOT / 'checkpoints'
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('DATA_DIR =', DATA_DIR)
print('OUT_DIR  =', OUT_DIR)

## 2.1 Validate / prepare CIC-IDS2017 files from Drive

If you already uploaded extracted CSVs or MachineLearningCSV.zip to Drive, this cell uses them and avoids re-download.

In [ ]:
import zipfile
from pathlib import Path

ALL_CIC_FILES = [
    'Monday-WorkingHours.pcap_ISCX.csv',
    'Tuesday-WorkingHours.pcap_ISCX.csv',
    'Wednesday-workingHours.pcap_ISCX.csv',
    'Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv',
    'Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv',
    'Friday-WorkingHours-Morning.pcap_ISCX.csv',
    'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv',
    'Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv',
]
EXPECTED_CIC_FILES = CIC_FILES if CIC_FILES else ALL_CIC_FILES

def _missing_cic_files(dst: Path) -> list[str]:
    return [name for name in EXPECTED_CIC_FILES if not (dst / name).exists()]

def _extract_zip_to_data_dir(zip_path: Path, dst: Path) -> int:
    extracted = 0
    with zipfile.ZipFile(zip_path) as zf:
        for member in zf.namelist():
            base = Path(member).name
            if not base.lower().endswith('.csv'):
                continue
            out = dst / base
            if out.exists():
                continue
            with zf.open(member) as src, out.open('wb') as f:
                f.write(src.read())
            extracted += 1
    return extracted

missing_before = _missing_cic_files(DATA_DIR)
print('Expecting', len(EXPECTED_CIC_FILES), 'CIC day files for this run')
print('CIC files already present:', len(EXPECTED_CIC_FILES) - len(missing_before), '/', len(EXPECTED_CIC_FILES))

if missing_before:
    zip_candidates = []
    for root in [DATA_DIR, DRIVE_ROOT]:
        if root.exists():
            zip_candidates.extend(sorted(root.glob('MachineLearningCSV*.zip')))
            zip_candidates.extend(sorted(root.glob('_MachineLearningCSV*.zip')))

    used_zip = None
    for zp in zip_candidates:
        try:
            n = _extract_zip_to_data_dir(zp, DATA_DIR)
            used_zip = zp
            print(f'Used zip: {zp} (newly extracted {n} CSV files)')
            break
        except zipfile.BadZipFile:
            print(f'Skipping invalid zip: {zp}')

missing_after = _missing_cic_files(DATA_DIR)
if missing_after:
    print('Still missing files required for this run:', missing_after)
    print('Training will try auto-download via mirrors as fallback.')
else:
    print('CIC-IDS2017 files ready in', DATA_DIR)


## 3. Sanity check GPU

In [ ]:
import torch
print('torch =', torch.__version__, 'cuda =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device =', torch.cuda.get_device_name(0))

## 4. Train

If the run stops mid-way: set `RESUME_CKPT` in §0 (e.g. last `ft_svdd_mfm_last.pt` or `ft_svdd_svdd_last.pt` under `OUT_DIR`), optionally `SKIP_MFM = True` if resuming after MFM finished, then re-run from this cell.

In [ ]:
import os
import runpy

# Line-buffer stdout so each epoch line appears immediately in Colab while the cell runs
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(line_buffering=True)

repo = Path.cwd()
train_py = repo / 'scripts' / 'train_phase2.py'
if not train_py.is_file():
    raise FileNotFoundError(
        f'Missing {train_py}. Push the scripts/ folder to GitHub and re-run the clone cell.'
    )

train_argv = [
    '--data-dir',
    str(DATA_DIR),
    '--out-dir',
    str(OUT_DIR),
    '--held-out-family',
    HELD_OUT_FAMILY,
    '--epochs-mfm',
    str(EPOCHS_MFM),
    '--epochs-svdd',
    str(EPOCHS_SVDD),
    '--batch-size',
    str(BATCH_SIZE),
    '--device',
    'auto',
]
if MAX_TRAIN_ROWS:
    train_argv += ['--max-train-rows', str(MAX_TRAIN_ROWS)]
if CIC_FILES:
    train_argv += ['--cic-files'] + CIC_FILES
if RESUME_CKPT:
    train_argv += ['--resume', str(RESUME_CKPT)]
if SKIP_MFM:
    train_argv.append('--skip-mfm')

print('LOW_RAM_MODE =', LOW_RAM_MODE)
print('BATCH_SIZE =', BATCH_SIZE, 'MAX_TRAIN_ROWS =', MAX_TRAIN_ROWS)
if CIC_FILES:
    print('CIC_FILES subset =', CIC_FILES)

_saved_argv = sys.argv[:]
try:
    os.chdir(repo)
    sys.argv = [str(train_py)] + train_argv
    runpy.run_path(str(train_py), run_name='__main__')
except SystemExit as exc:
    if exc.code not in (0, None):
        raise RuntimeError(
            f'train_phase2.py exited with code {exc.code}. '
            'Scroll up for the traceback. Common Colab fixes: enable GPU runtime; '
            'lower BATCH_SIZE (e.g. 256) or set smaller MAX_TRAIN_ROWS; '
            'set LOW_RAM_MODE = True; ensure Drive has free space; '
            'retry if CIC-IDS2017 download failed.'
        ) from exc
finally:
    sys.argv = _saved_argv

## 5. Plot learning curves (sanity check before pulling artifacts)

In [ ]:
import json, matplotlib.pyplot as plt
hist = json.loads((OUT_DIR / 'history.json').read_text())
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(hist['pretrain_train'], label='train')
ax[0].plot(hist['pretrain_val'], label='val')
ax[0].set_title('MFM masked-MSE'); ax[0].set_xlabel('epoch'); ax[0].legend()
ax[1].plot(hist['finetune_loss'], label='svdd loss')
ax[1].plot(hist['finetune_radius'], label='mean radius')
ax[1].set_title('Deep SVDD finetune'); ax[1].set_xlabel('epoch'); ax[1].legend()
plt.tight_layout(); plt.show()

## 6. (Optional) zip artifacts for `git push` of the metrics JSON

Weights stay in Drive. We only `git push` the small JSON / PNG artifacts so the repo
stays lightweight.

In [ ]:
for f in sorted(OUT_DIR.iterdir()):
    print(f.name, '%.1f KB' % (f.stat().st_size / 1024))